In [ ]:
# ============================================================
# AI-POWERED CYBERBULLYING & TOXICITY DETECTION SYSTEM
# ALL-IN-ONE GOOGLE COLAB CODE
# ============================================================

import subprocess
import sys

# STEP 1: Install libraries
print("Installing required libraries...")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers", "torch", "gradio",
    "pandas", "matplotlib"
])

# STEP 2: Import libraries
import torch
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt
import tempfile
import os

from transformers import AutoTokenizer, AutoModelForSequenceClassification

# STEP 3: Load pretrained AI model
print("Loading Cyberbullying Detection AI Model...")

MODEL_NAME = "unitary/toxic-bert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("AI Model Loaded Successfully!")
print("Device:", device)

LABELS = [
    "Toxic",
    "Severe Toxic",
    "Obscene",
    "Threat",
    "Insult",
    "Identity Hate"
]

# STEP 4: Prediction function
def predict_toxicity(text):

    if not isinstance(text, str) or not text.strip():
        return None

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.sigmoid(outputs.logits)[0].cpu().numpy()

    scores = {
        label: round(float(score) * 100, 2)
        for label, score in zip(LABELS, probabilities)
    }

    maximum = max(scores.values())

    if maximum >= 70:
        status = "HIGH TOXICITY"
    elif maximum >= 40:
        status = "MODERATE TOXICITY"
    else:
        status = "LOW TOXICITY"

    return scores, status


# STEP 5: Analyze individual comment
def analyze_text(text):

    result = predict_toxicity(text)

    if result is None:
        return "Please enter a comment.", None, pd.DataFrame()

    scores, status = result

    if status == "HIGH TOXICITY":
        message = "🚨 HIGH TOXICITY DETECTED"
    elif status == "MODERATE TOXICITY":
        message = "⚠️ MODERATE TOXICITY DETECTED"
    else:
        message = "✅ LOW TOXICITY DETECTED"

    # Generate graph
    fig, ax = plt.subplots(figsize=(9, 5))

    colors = [
        "red" if score >= 70
        else "orange" if score >= 40
        else "green"
        for score in scores.values()
    ]

    bars = ax.bar(
        list(scores.keys()),
        list(scores.values()),
        color=colors
    )

    ax.set_title("AI Toxicity Analysis", fontsize=15)
    ax.set_ylabel("Toxicity Score (%)")
    ax.set_ylim(0, 110)

    ax.tick_params(axis="x", rotation=25)

    for bar, score in zip(bars, scores.values()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            score + 2,
            f"{score:.1f}%",
            ha="center"
        )

    plt.tight_layout()

    # Detailed table
    table = pd.DataFrame({
        "Toxicity Category": list(scores.keys()),
        "Score (%)": list(scores.values())
    })

    return message, fig, table


# STEP 6: CSV batch analysis
def analyze_csv(file_path):

    if file_path is None:
        return "Please upload a CSV file.", None, None

    try:
        df = pd.read_csv(file_path)

        # Accept common column names
        possible_columns = [
            "comment", "text", "message",
            "Comment", "Text", "Message"
        ]

        text_column = next(
            (col for col in possible_columns if col in df.columns),
            None
        )

        if text_column is None:
            return (
                "CSV must contain a 'comment' or 'text' column.",
                None,
                None
            )

        results = []

        for comment in df[text_column].fillna("").astype(str):

            prediction = predict_toxicity(comment)

            if prediction is None:
                results.append({
                    "Status": "EMPTY TEXT",
                    **{label: 0 for label in LABELS}
                })
                continue

            scores, status = prediction

            results.append({
                "Status": status,
                **scores
            })

        results_df = pd.DataFrame(results)

        final_df = pd.concat(
            [df.reset_index(drop=True), results_df],
            axis=1
        )

        output_path = os.path.join(
            tempfile.gettempdir(),
            "cyberbullying_analysis_results.csv"
        )

        final_df.to_csv(output_path, index=False)

        return (
            f"Successfully analyzed {len(df)} comments!",
            final_df,
            output_path
        )

    except Exception as e:
        return f"Error: {str(e)}", None, None


# STEP 7: Create Gradio web interface
with gr.Blocks(
    title="AI Cyberbullying Detection System",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("""
    # 🛡️ AI-POWERED CYBERBULLYING DETECTION SYSTEM

    ### Advanced Text and Sentiment Analysis Project

    **Technology:** Python | NLP | BERT | Deep Learning

    Detect potentially harmful language in online comments
    using Artificial Intelligence.

    ---
    """)

    with gr.Tabs():

        # TAB 1: Individual text analysis
        with gr.Tab("🔍 Text Analysis"):

            gr.Markdown("## Enter a Comment for Analysis")

            text_input = gr.Textbox(
                label="Enter Text",
                placeholder="Type your comment here...",
                lines=5
            )

            analyze_btn = gr.Button(
                "🚀 Analyze Comment",
                variant="primary"
            )

            result_output = gr.Textbox(
                label="AI Detection Result"
            )

            graph_output = gr.Plot(
                label="Toxicity Visualization"
            )

            table_output = gr.Dataframe(
                label="Detailed Analysis"
            )

            analyze_btn.click(
                fn=analyze_text,
                inputs=text_input,
                outputs=[
                    result_output,
                    graph_output,
                    table_output
                ]
            )

            gr.Examples(
                examples=[
                    ["You are an amazing person!"],
                    ["You are stupid and useless."],
                    ["I will hurt you."],
                    ["Thank you for helping me."],
                    ["Nobody likes you."]
                ],
                inputs=text_input
            )

        # TAB 2: CSV analysis
        with gr.Tab("📁 CSV Analysis"):

            gr.Markdown("""
            ## Upload CSV Dataset

            Upload a CSV file containing a column named
            **comment**, **text**, or **message**.
            """)

            csv_input = gr.File(
                label="Upload CSV File",
                file_types=[".csv"],
                type="filepath"
            )

            csv_button = gr.Button(
                "📊 Analyze Dataset",
                variant="primary"
            )

            csv_status = gr.Textbox(
                label="Analysis Status"
            )

            csv_table = gr.Dataframe(
                label="Dataset Results"
            )

            csv_download = gr.File(
                label="Download Analysis Results"
            )

            csv_button.click(
                fn=analyze_csv,
                inputs=csv_input,
                outputs=[
                    csv_status,
                    csv_table,
                    csv_download
                ]
            )

        # TAB 3: About
        with gr.Tab("ℹ️ About Project"):

            gr.Markdown("""
            ## AI Cyberbullying Detection System

            ### Project Objective
            Identify potentially toxic or abusive online
            comments using Natural Language Processing.

            ### Detection Categories
            1. Toxic Language
            2. Severe Toxicity
            3. Obscene Language
            4. Threats
            5. Insults
            6. Identity-Based Hate

            ### Technologies
            - Python
            - Google Colab
            - BERT Transformer
            - PyTorch
            - Hugging Face Transformers
            - Pandas
            - Matplotlib
            - Gradio

            ### Applications
            - Social media comment moderation
            - Online community safety
            - Harmful language research
            - Toxic comment classification

            ### Important Limitations
            This is primarily an English-language toxicity
            classifier, not a complete cyberbullying detector.

            It cannot reliably determine repeated harassment
            or the full context of a conversation.

            The LOW, MODERATE, and HIGH thresholds are
            illustrative and are not validated safety ratings.
            """)

    gr.Markdown("""
    ---
    ### 🎓 Advanced TSA Mini Project
    **AI-Powered Cyberbullying and Toxicity Detection
    Using NLP and Deep Learning**
    """)


# STEP 8: Launch application
print("\n" + "=" * 55)
print("CYBERBULLYING DETECTION SYSTEM READY!")
print("=" * 55)

demo.launch(share=True, debug=True)

Installing required libraries...
Loading Cyberbullying Detection AI Model...


config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

AI Model Loaded Successfully!
Device: cpu


/tmp/ipykernel_1108/1558268748.py:218: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(



CYBERBULLYING DETECTION SYSTEM READY!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://afddb74be08d367d32.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
